# E-Commerce Conversion Prediction - Summer Analytics 2026
## Week 2 Hackathon Solution
**Model:** Ensemble of Random Forest + Gradient Boosting + Extra Trees
**Metric:** F1 Score

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 1. Load Data

In [ ]:
train = pd.read_csv('train.csv')
pub   = pd.read_csv('public_test.csv')
priv  = pd.read_csv('private_test.csv')

print('Train shape:', train.shape)
print('Public Test shape:', pub.shape)
print('Private Test shape:', priv.shape)
train.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Target Distribution ===')
print(train['Converted'].value_counts())
print(f'\nClass imbalance ratio: {(train["Converted"]==0).sum()} : {(train["Converted"]==1).sum()}')

print('\n=== Missing Values ===')
print(train.isnull().sum()[train.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Analysis', fontsize=14)

# Conversion by device
train.groupby('Device_Type')['Converted'].mean().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Conversion Rate by Device')
axes[0,0].set_ylabel('Conversion Rate')

# Conversion by traffic source
train.groupby('Traffic_Source')['Converted'].mean().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Conversion Rate by Traffic Source')

# Age distribution
train[train['Converted']==1]['Age'].hist(ax=axes[0,2], alpha=0.6, label='Converted', bins=30, color='green')
train[train['Converted']==0]['Age'].hist(ax=axes[0,2], alpha=0.6, label='Not Converted', bins=30, color='red')
axes[0,2].set_title('Age Distribution by Conversion')
axes[0,2].legend()

# Pages viewed
train.boxplot(column='Pages_Viewed', by='Converted', ax=axes[1,0])
axes[1,0].set_title('Pages Viewed by Conversion')

# Previous purchases
train.boxplot(column='Previous_Purchases', by='Converted', ax=axes[1,1])
axes[1,1].set_title('Previous Purchases by Conversion')

# Discount seen
train.groupby(['Discount_Seen', 'Converted']).size().unstack().plot(kind='bar', ax=axes[1,2])
axes[1,2].set_title('Discount vs Conversion')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('EDA complete!')

## 3. Feature Engineering & Preprocessing

In [ ]:
def preprocess(df, fit_encoders=None, fit_imputer_num=None):
    df = df.copy()
    
    # --- Feature Engineering ---
    # Interaction features
    df['Pages_per_Product']          = df['Pages_Viewed'] / (df['Products_Viewed'] + 1)
    df['Time_per_Page']              = df['Time_On_Site'] / (df['Pages_Viewed'] + 1)
    df['Engagement_Score']           = df['Pages_Viewed'] * df['Time_On_Site'] / (df['Products_Viewed'] + 1)
    df['Purchase_Discount_Interaction'] = df['Previous_Purchases'] * df['Discount_Seen']
    df['High_Value_User']            = ((df['Income'] > df['Income'].median()) & 
                                         (df['Previous_Purchases'] > 2)).astype(int)
    df['Log_Income']                 = np.log1p(df['Income'].fillna(df['Income'].median()))
    df['Age_Group']                  = pd.cut(df['Age'].fillna(df['Age'].median()), 
                                               bins=[0,25,35,50,100], labels=[0,1,2,3]).astype(float)
    
    cat_cols = ['Device_Type', 'Traffic_Source']
    num_cols = ['Age', 'Income', 'Time_On_Site', 'Pages_per_Product', 
                'Time_per_Page', 'Engagement_Score', 'Log_Income', 'Age_Group']
    
    # Label Encoding
    if fit_encoders is None:
        fit_encoders = {}
        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            fit_encoders[col] = le
    else:
        for col in cat_cols:
            le = fit_encoders[col]
            df[col] = df[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    
    # Imputation
    if fit_imputer_num is None:
        fit_imputer_num = SimpleImputer(strategy='median')
        df[num_cols] = fit_imputer_num.fit_transform(df[num_cols])
    else:
        df[num_cols] = fit_imputer_num.transform(df[num_cols])
    
    feature_cols = ['Age', 'Income', 'City_Tier', 'Device_Type', 'Traffic_Source',
                    'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases',
                    'Discount_Seen', 'Browser_Version', 'Campaign_Code',
                    'Pages_per_Product', 'Time_per_Page', 'Engagement_Score',
                    'Purchase_Discount_Interaction', 'High_Value_User', 'Log_Income', 'Age_Group']
    
    return df[feature_cols], fit_encoders, fit_imputer_num

X, encoders, imputer_num = preprocess(train)
y = train['Converted']
print('Feature engineering done! Final feature count:', X.shape[1])

## 4. Model Training - Ensemble Approach

In [ ]:
# Define individual models
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
et = ExtraTreesClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)

# Cross-validation
print('Running 5-fold Cross Validation...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Random Forest', rf), ('Gradient Boosting', gb), ('Extra Trees', et)]:
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    print(f'{name}: Mean F1 = {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# Train on full training data
print('Training models on full dataset...')
rf.fit(X, y)
gb.fit(X, y)
et.fit(X, y)
print('All models trained!')

## 5. Threshold Optimization on Public Test

In [ ]:
X_pub, _, _ = preprocess(pub, encoders, imputer_num)
y_pub = pub['Converted']

# Ensemble probabilities (weighted)
prob_rf  = rf.predict_proba(X_pub)[:,1]
prob_gb  = gb.predict_proba(X_pub)[:,1]
prob_et  = et.predict_proba(X_pub)[:,1]
prob_ens = (prob_rf * 0.35 + prob_gb * 0.35 + prob_et * 0.30)

# Find best threshold
best_thresh, best_f1 = 0.5, 0
f1_scores = []
thresholds = np.arange(0.2, 0.8, 0.01)

for t in thresholds:
    preds = (prob_ens >= t).astype(int)
    f = f1_score(y_pub, preds)
    f1_scores.append(f)
    if f > best_f1:
        best_f1 = f
        best_thresh = t

# Plot
plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best threshold = {best_thresh:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold on Public Test')
plt.legend()
plt.tight_layout()
plt.savefig('threshold_plot.png', dpi=100)
plt.show()

print(f'Best Threshold: {best_thresh:.2f}')
print(f'Best Public F1 Score: {best_f1:.4f}')

# Final eval on public
final_pub_preds = (prob_ens >= best_thresh).astype(int)
print('\n=== Public Test Classification Report ===')
print(classification_report(y_pub, final_pub_preds))

## 6. Feature Importance

In [ ]:
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(12), x='Importance', y='Feature', palette='viridis')
plt.title('Top 12 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()
print(feat_imp.head(10).to_string(index=False))

## 7. Generate Final Submission

In [ ]:
X_priv, _, _ = preprocess(priv, encoders, imputer_num)

prob_priv_rf  = rf.predict_proba(X_priv)[:,1]
prob_priv_gb  = gb.predict_proba(X_priv)[:,1]
prob_priv_et  = et.predict_proba(X_priv)[:,1]
prob_priv     = (prob_priv_rf * 0.35 + prob_priv_gb * 0.35 + prob_priv_et * 0.30)

final_preds = (prob_priv >= best_thresh).astype(int)

submission = pd.read_csv('sample_submission.csv')
submission['Converted'] = final_preds
submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print('Prediction distribution:')
print(pd.Series(final_preds).value_counts())
print('\nFirst 5 rows:')
print(submission.head())